# Cockpit (prototype)

Hier praat je als PO met de harness. Voorstellen en vragen verschijnen als kaarten onder de cel **Cockpit**.

Dit prototype gebruikt nog geen LLM-agents. Het test de basis: kaarten tonen, op jouw besluit wachten en het besluit vastleggen in het beslislog.

1. Voer alle cellen uit.
2. Beantwoord de vraag en de voorstellen in de cockpit.
3. Voer daarna de laatste cel uit om het beslislog te zien.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from dsharness.cockpit import Cockpit
from dsharness.beslislog import Beslislog
from dsharness.voorbeeld import voorbeeld_data, voorbeeld_voorstel

Path("beslislog_demo.jsonl").unlink(missing_ok=True)  # demo: elke keer opnieuw beginnen
log = Beslislog("beslislog_demo.jsonl")

## Cockpit

In [2]:
c = Cockpit(log)
c

VBox()

## Demo-workflow

Draait als achtergrondtaak (`c.start`). De cel is dus meteen klaar, terwijl de workflow in de cockpit op jou wacht.

In [3]:
async def demo():
    df = voorbeeld_data()
    n_na = int(df["aanrijtijd"].isna().sum())
    c.melding(f"Data Scout: {n_na} NA's in 'aanrijtijd'. NA-patroon onderzocht: geen verschil tussen regio's.")

    antwoord = await c.vraag_po(
        "Weet je waarom 'aanrijtijd' soms ontbreekt?",
        f"{n_na} van de {len(df)} ritten ({n_na / len(df):.0%}) hebben geen aanrijtijd.",
    )

    fig, ax = plt.subplots(figsize=(5, 2.4))
    ax.hist(df["aanrijtijd"].dropna(), bins=40, color="#2F6FAF")
    ax.axvline(df["aanrijtijd"].median(), color="#2C7A57", ls="--", label="mediaan")
    ax.axvline(df["aanrijtijd"].mean(), color="#B42318", label="gemiddelde")
    ax.set_xlabel("aanrijtijd (min)"); ax.legend(frameon=False)
    plt.close(fig)

    v1 = voorbeeld_voorstel("twijfel").model_copy(update={"po_kennis": f"PO: {antwoord}"})
    b1 = await c.vraag_akkoord(v1, figuur=fig)
    c.melding(f"Besluit {b1.voorstel_id} vastgelegd: {b1.keuze}.")

    v2 = voorbeeld_voorstel("fout").model_copy(update={"id": "F2-003", "fase": "F2 Regels (vóór de split)"})
    b2 = await c.vraag_akkoord(v2)
    c.melding(f"Besluit {b2.voorstel_id} vastgelegd: {b2.keuze}. Demo klaar.")

taak = c.start(demo())

## Beslislog

In [4]:
pd.DataFrame(
    [{"voorstel": v.id, "actie": v.actie, "reviewer": v.review.oordeel, "besluit": b.keuze, "motivatie": b.motivatie, "tijd": b.tijdstip}
     for v, b in log.lees()]
)

""
